# Lesson 4 — Put the MCP order assistant behind FastAPI

**Goal:** turn Lesson 3 into a small, real HTTP application with clear API, service, repository, validation, and dependency-injection boundaries.

The finished application is in `lesson_4_app/`. The notebook explains only the decisions needed to understand and run it.

## 0. Setup

The application still needs the Lesson 2 MCP server. FastAPI listens on port `8001`; the MCP server listens on `8000`.

In [1]:
# Run once if needed
%pip install -qU fastapi==0.141.1 langchain-core==1.6.3 langchain-mcp-adapters==0.3.2 langchain-openai==1.6.2 openai==3.14.1 pydantic==2.13.5 python-dotenv==1.2.3 PyYAML==6.0.3 "uvicorn[standard]==0.53.0"

Note: you may need to restart the kernel to use updated packages.


### Require the OpenAI key immediately

The notebook and application fail early when `.env` or `OPENAI_API_KEY` is missing. A configuration error should not wait until the first customer request.

In [2]:
from pathlib import Path

from dotenv import dotenv_values, load_dotenv


dotenv_path = Path.cwd() / ".env"
if not dotenv_path.is_file():
    raise FileNotFoundError(
        f"Missing {dotenv_path}. Create it with OPENAI_API_KEY=your-key."
    )
if not dotenv_values(dotenv_path).get("OPENAI_API_KEY"):
    raise RuntimeError(f"OPENAI_API_KEY is missing or empty in {dotenv_path}.")
load_dotenv(dotenv_path, override=True)
print("Environment loaded.")

Environment loaded.


# 1. Layered architecture—quickly

A layer has one reason to change and depends inward on application logic:

```text
HTTP request
    ↓
API / route       validates HTTP input and returns HTTP output
    ↓
Service           runs the use case and coordinates the LLM
    ↓
Repository        hides MCP connection and tool discovery
    ↓
FastMCP server    accesses trusted order data
```

| Layer | Contains | Must not contain |
|---|---|---|
| API/route | URL, HTTP method, request/response models | Prompt orchestration or MCP transport code |
| Service | The `answer order question` use case | FastAPI request objects |
| Repository | External data/tool access | HTTP routing or user-facing response wording |
| Schema | Typed boundary contracts | Business workflows |

For a tiny app these could fit in one file. We separate them here because the boundaries are the lesson; each file remains small.

## Minimal project structure

```text
lesson_4_app/
├── app/
│   ├── main.py          # creates FastAPI and includes the router
│   ├── api/routes.py    # HTTP endpoint
│   ├── schemas.py       # request and response contracts
│   ├── services.py      # LLM + tool-calling use case
│   ├── prompt_loader.py # loads and validates YAML prompts
│   ├── prompts/
│   │   └── order_assistant.yml
│   ├── repositories.py  # MCP boundary
│   ├── dependencies.py  # object construction with Depends
│   └── config.py        # strict environment loading
└── requirements.txt
```

FastAPI's `APIRouter` keeps related path operations in a module, while `app.include_router(...)` assembles them into one application.

# 2. Transform Lesson 3 into these layers

| Lesson 3 responsibility | Lesson 4 location |
|---|---|
| `SupportAnswer` | `schemas.py` |
| Question input | `OrderQuestion` in `schemas.py` |
| MCP connection and discovery | `OrderToolRepository` |
| Prompt text | `prompts/order_assistant.yml` |
| Prompt loading and validation | `prompt_loader.py` |
| Model tool choice, tool result, structured answer | `OrderAssistantService` |
| Object creation | `dependencies.py` |
| Public HTTP endpoint | `api/routes.py` |
| Application assembly | `main.py` |

Nothing important is rewritten: the Lesson 3 workflow is moved behind stable boundaries.

### Compatibility choices used here

- `ChatOpenAI(use_responses_api=True)` selects OpenAI's Responses API explicitly, which is the recommended tool-calling path and avoids endpoint ambiguity.
- Shared model construction does not send model-specific `reasoning_effort` or `temperature` arguments. This keeps both GPT-4o and reasoning models valid; add such controls only after selecting a compatible model.
- `langchain-mcp-adapters` and `MultiServerMCPClient` provide the stable LangChain/MCP boundary. The older `langchain.mcp.MCPAdapter` path is beta.
- Exact dependency versions make the notebook and Docker image reproducible. Upgrade them intentionally and retest the `/ask` flow.

## Input and output validation

There are three validation boundaries:

1. **HTTP input:** `OrderQuestion` rejects missing, blank, too-short, or oversized questions before the service runs. Invalid requests receive FastAPI's `422` response.
2. **LLM output:** `with_structured_output(SupportAnswer)` asks the model for the exact Pydantic schema and parses it into that model.
3. **HTTP output:** `response_model=SupportAnswer` makes FastAPI validate, document, serialize, and filter the public response.

Do not trust free-form model text as an API contract. Validate model output before it crosses the HTTP boundary.

## Keep prompts outside Python

Production prompts change independently from orchestration code. `app/prompts/order_assistant.yml` therefore owns the system and final-answer instructions. This makes prompt reviews and version-control diffs focused and avoids redeploy-oriented code edits for wording changes.

The application uses `yaml.safe_load`—never the unsafe generic loader—and validates the result with `OrderPrompts`. `get_order_prompts()` is cached and called during startup, so a missing file, malformed YAML, or missing prompt field fails before traffic is accepted. The validated object is then injected into `OrderAssistantService`.

In [3]:
from typing import Literal

from pydantic import BaseModel, ConfigDict, Field


class OrderQuestion(BaseModel):
    model_config = ConfigDict(str_strip_whitespace=True)
    question: str = Field(min_length=3, max_length=500)


class SupportAnswer(BaseModel):
    answer: str = Field(min_length=1)
    status: Literal["shipped", "processing", "not_found", "unknown"]
    grounded: bool

# 3. Dependency injection: why, then how

A route needs an `OrderAssistantService`, but it should not know how to create an OpenAI client, load prompts, read configuration, or connect to MCP. **Dependency injection (DI)** lets the route declare what it needs; FastAPI builds and supplies it.

Why it matters:

- construction exists in one place;
- shared configuration is not repeated;
- FastAPI resolves the dependency graph once per request;
- tests can replace the service or repository without calling OpenAI or MCP.

`Depends(get_order_service)` tells FastAPI **how to obtain the value**. `Annotated[OrderAssistantService, ...]` preserves its Python type for editors and type checkers. A reusable alias keeps route signatures short.

In [ ]:
from typing import Annotated

from fastapi import Depends


# FastAPI calls this provider; the route does not call it.
def get_order_service() -> OrderAssistantService:
    ...


OrderServiceDep = Annotated[
    OrderAssistantService,
    Depends(get_order_service),
]


# The real route can now state only what it needs:
# async def ask_order(request: OrderQuestion, service: OrderServiceDep): ...

# 4. The complete minimal application

The real implementation is in `lesson_4_app/`; it contains no placeholder application code. Start the MCP server and API from the repository root in separate terminals:

```bash
python order_mcp_server.py
```

```bash
uvicorn lesson_4_app.app.main:app --reload --port 8001
```

Open `http://127.0.0.1:8001/docs`, or send:

```bash
curl -X POST http://127.0.0.1:8001/api/v1/orders/ask \
  -H 'Content-Type: application/json' \
  -d '{"question":"What is happening with order A100?"}'
```

## References

- [FastAPI: bigger applications and APIRouter](https://fastapi.tiangolo.com/tutorial/bigger-applications/)
- [FastAPI: dependencies and Annotated](https://fastapi.tiangolo.com/tutorial/dependencies/)
- [FastAPI: response models](https://fastapi.tiangolo.com/tutorial/response-model/)
- [OpenAI Responses API](https://developers.openai.com/api/reference/cli/resources/responses/methods/create)

# 5. All required code in one overview

This final cell prints the actual application files in dependency order. Edit the files—not a duplicate notebook copy.

In [5]:
from pathlib import Path


app_root = Path("lesson_4_app")
required_files = [
    "app/schemas.py",
    "app/config.py",
    "app/prompts/order_assistant.yml",
    "app/prompt_loader.py",
    "app/repositories.py",
    "app/services.py",
    "app/dependencies.py",
    "app/api/routes.py",
    "app/main.py",
    "requirements.txt",
]

for relative_path in required_files:
    path = app_root / relative_path
    if not path.is_file():
        raise FileNotFoundError(path)
    print(f"\n{'=' * 16} {path} {'=' * 16}\n")
    print(path.read_text())


================ lesson_4_app/app/schemas.py ================

from typing import Literal

from pydantic import BaseModel, ConfigDict, Field


class OrderQuestion(BaseModel):
    """Validated HTTP input."""

    model_config = ConfigDict(str_strip_whitespace=True)

    question: str = Field(min_length=3, max_length=500)


class SupportAnswer(BaseModel):
    """Validated LLM and HTTP output."""

    answer: str = Field(min_length=1)
    status: Literal["shipped", "processing", "not_found", "unknown"]
    grounded: bool = Field(
        description="True only when the status came from a successful MCP tool call"
    )


================ lesson_4_app/app/config.py ================

import os
from dataclasses import dataclass, field
from functools import lru_cache
from pathlib import Path

from dotenv import dotenv_values, load_dotenv


@dataclass(frozen=True)
class Settings:
    openai_api_key: str = field(repr=False)
    mcp_url: str = "http://127.0.0.1:8000/mcp"
    model: str = "gpt-5